Import Libraries

In [ ]:
from typing import TypedDict, Annotated, List
from operator import add
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv

load_dotenv()
print("Imports are Successfull!!")

Initialize Claude

In [ ]:
llm = ChatAnthropic(
    model = "claude-sonnet-4-6",
    temperature = 0
)    

Define AgentState

In [ ]:
class AgentState(TypedDict):
    infrastructure: str
    audit_brief: str
    audit_reports: Annotated[List[str], add]
    final_report: str

print("Agent State defined!!")

Define Orchestrator Node.

In [ ]:
def orchestrator(state: AgentState) -> dict:
    print("\n Orchestrator preparing audit brief...")

    response = llm.invoke([
    SystemMessage(content="""You are a senior cloud security audit coordinator.
        Your job is to analyze an infrastructure description and prepare a clear, 
        concise audit brief that will guide three specialist security auditors.
        The brief should identify the key systems, components, and risk areas 
        that need to be examined. Keep it under 200 words."""),
    HumanMessage(content=f"Infrastructure to audit:\n{state['infrastructure']}")
    ])

    print("Audit brief ready")
    return {"audit_brief": response.content}

print("Orchestrator node defined!")

Specialist Audit Nodes

In [ ]:
def iam_audit(state: AgentState) -> dict:
    print("\n IAM Audit Agent auditing...")
    response = llm.invoke([
        SystemMessage(content="""You are an IAM security specialist.
        Audit the infrastructure for Identity and Access Management risks.
        Cover: authentication, authorization, least privilege, MFA, 
        service accounts, and role assignments.
        Be specific and actionable. Maximum 200 words."""),
        HumanMessage(content=f"Audit brief:\n{state['audit_brief']}\n\nInfrastructure:\n{state['infrastructure']}")
    ])
    print("IAM audit complete")
    return {"audit_reports": [f"=== IAM AUDIT ===\n{response.content}"]}

def network_audit(state: AgentState) -> dict:
    print("\n Network Audit Agent auditing...")
    response = llm.invoke([
        SystemMessage(content="""You are a network security specialist.
        Audit the infrastructure for network security risks.
        Cover: firewall rules, network segmentation, VPC configuration,
        exposed ports, traffic encryption, and DDoS protection.
        Be specific and actionable. Maximum 200 words."""),
        HumanMessage(content=f"Audit brief:\n{state['audit_brief']}\n\nInfrastructure:\n{state['infrastructure']}")
    ])
    print("Network audit complete")
    return {"audit_reports": [f"=== NETWORK AUDIT ===\n{response.content}"]}

def compliance_audit(state: AgentState) -> dict:
    print("\n Compliance AuditAgent auditing...")
    response = llm.invoke([
        SystemMessage(content="""You are a cloud compliance specialist.
        Audit the infrastructure for compliance and governance risks.
        Cover: regulatory requirements, data protection, logging,
        monitoring, patch management, and security benchmarks like
        CIS and NIST. Be specific and actionable. Maximum 200 words."""),
        HumanMessage(content=f"Audit brief:\n{state['audit_brief']}\n\nInfrastructure:\n{state['infrastructure']}")
    ])
    print("Compliance audit complete")
    return {"audit_reports": [f"=== COMPLIANCE AUDIT ===\n{response.content}"]}

print("All three audit agents defined!!")

Synthesizer Node

In [ ]:
def synthesizer(state: AgentState) -> dict:
    print("\n Synthesizer combining all audit reports..")
    
    all_reports = "\n\n".join(state["audit_reports"])

    response = llm.invoke([
        SystemMessage(content="""You are a senior security audit manager.
        Combine the three specialist audit reports into one comprehensive,
        executive-level security assessment. Highlight the most critical
        findings, common themes across all three audits, and provide
        a prioritized list of recommended actions. Maximum 400 words."""),
        HumanMessage(content=f"""
Audit Brief:
{state['audit_brief']}

Individual Audit Reports:
{all_reports}

Please synthesize these into a final comprehensive security assessment.""")
    ])
    
    print("Final report ready")
    return {"final_report": response.content}

print("Synthesizer node defined!")

Building Graph

In [ ]:
graph = StateGraph(AgentState)

#Nodes
graph.add_node("orchestrator", orchestrator)

graph.add_node("iam_audit", iam_audit)
graph.add_node("network_audit", network_audit)
graph.add_node("compliance_audit", compliance_audit)

graph.add_node("synthesizer", synthesizer)

#Edges
graph.add_edge(START, "orchestrator")

graph.add_edge("orchestrator", "iam_audit")
graph.add_edge("orchestrator", "network_audit")
graph.add_edge("orchestrator", "compliance_audit")

graph.add_edge("iam_audit", "synthesizer")
graph.add_edge("network_audit", "synthesizer")
graph.add_edge("compliance_audit", "synthesizer")

graph.add_edge("synthesizer", END)

#Compile
app = graph.compile()
print("Graph built and compiled!!")

Invoking the Graph

In [ ]:
result = app.invoke({
    "infrastructure" : "Create Kubernetes Cluster in Azure in US West Region",
    "audit_brief": "",
    "audit_reports": [],
    "final_report": ""
})

print("\n" + "="*60)
print("AUDIT BRIEF:")
print("="*60)
print(result["audit_brief"])

print("\n" + "="*60)
print("INDIVIDUAL AUDIT REPORTS:")
print("="*60)
for report in result["audit_reports"]:
    print(report)
    print()

print("\n" + "="*60)
print("FINAL SECURITY ASSESSMENT:")
print("="*60)
print(result["final_report"])